In [13]:
from pathlib import Path
from typing import List, Union, Any


In [5]:
vault_path = Path(r"C:\stuff\NCKH")

for file in vault_path.rglob("*.md"):
    print(file)

C:\stuff\NCKH\Adversarial Validation.md
C:\stuff\NCKH\Autoregressive Model.md
C:\stuff\NCKH\Causal Inference.md
C:\stuff\NCKH\Conditional Independence testing.md
C:\stuff\NCKH\Diagram.md
C:\stuff\NCKH\Financial Regime.md
C:\stuff\NCKH\Invariant Causal Prediction.md
C:\stuff\NCKH\Laplace Transformation.md
C:\stuff\NCKH\Math background.md
C:\stuff\NCKH\NCKH.md
C:\stuff\NCKH\Process macro indicators.md
C:\stuff\NCKH\Random Fourier Features.md
C:\stuff\NCKH\Robust Portfolio Construction.md
C:\stuff\NCKH\Sharpe ratio.md
C:\stuff\NCKH\Stationarity.md
C:\stuff\NCKH\Variables for defining Environment.md


In [14]:
def resolve_safe_path(vault_root: Path, input_path: Union[str, Path]) -> Path:
    """
    Chuẩn hóa và kiểm tra đường dẫn an toàn.
    Chặn đứng mọi kỹ thuật Path Traversal (../, absolute path, symlinks ngoài vault).
    """
    path = Path(input_path)

    if not path.is_absolute():
        path = vault_root / path

    full_path = path.resolve()

    if not full_path.is_relative_to(vault_root):
        raise PermissionError(f"Access to {full_path} is denied. Path traversal detected.")

    return full_path

## Function

Khi xây dựng ứng dụng (đặc biệt là hệ thống AI Agent / MCP server), hàm sẽ nhận dữ liệu từ hai nguồn:

- Từ bên ngoài (JSON / LLM / CLI / REST API): Dữ liệu truyền vào luôn ở dạng chuỗi (str), ví dụ: "vault/notes/mle.md".

- Từ nội bộ mã nguồn: Các hàm xử lý file khác thường trả về đối tượng pathlib.Path (chẳng hạn list_notes() trả về danh sách Path từ rglob)


Khai báo Union[str, Path] cho phép hàm chấp nhận cả hai mà không bắt người gọi phải bận tâm về kiểu dữ liệu.

In [15]:
class NoteManager:
    def __init__(self, vault_path):
        self.vault = Path(vault_path).resolve()
        self.vault.mkdir(parents=True, exist_ok=True)

    def list_notes(self) -> List[Path]:
        return list(self.vault.rglob("*.md"))

    def read_note(self, file_path: str | Path) -> str:
        path = resolve_safe_path(self.vault, file_path)
        return path.read_text(encoding='utf-8')

    def create_note(self, file_path: str | Path, content: str = "") -> None:
        path = resolve_safe_path(self.vault, file_path)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(content, encoding='utf-8')

    def update_note(self, file_path: str | Path, content: str) -> None:
        path = resolve_safe_path(self.vault, file_path)
        if not path.exists():
            self.create_note(path, content)
            return
        
        with path.open('a', encoding = 'utf-8') as f:
            if path.stat().st_size > 0 and not path.read_text(encoding='utf-8').endswith('\n'):
                f.write('\n')
            f.write(content)

    def delete_note(self, file_path: str | Path) -> None:
        path = resolve_safe_path(self.vault, file_path)
        if path.exists():
            path.unlink()

    def search_notes(self, keyword:str) -> List[Path]:
        results = []
        for note in self.list_notes():
            if keyword.lower() in note.read_text(encoding = 'utf-8').lower():
                results.append(note)
        return results

    
